In [1]:
#r "nuget: Deedle.Interactive"
#r "nuget: ARCtrl"
#r "nuget: FsHttp"

Installed Packages ARCtrl, 3.1.0 Deedle.Interactive, 3.0.0 FsHttp, 15.0.3

Loading extensions from `C:\Users\schne\.nuget\packages\deedle.interactive\3.0.0\lib\netstandard2.1\Deedle.Interactive.dll`

In [2]:
open Deedle

let df = Frame.ReadCsv("planned_phase_1.csv")   

let candidates = 
    df
    |> Frame.mapRows (fun rk os ->
        os.GetAs<string>("Project_Accession"), os.GetAs<int>("Total_Attributes")
    )
    |> Series.values
    |> Seq.sortByDescending (fun (_, total) -> total)
    |> Array.ofSeq
    |> Array.map fst
    |> Set

candidates

Count,50
(values),"[ PRJEB22060, PRJEB24450, PRJEB25079, PRJEB27774, PRJEB33339, PRJNA1014093, PRJNA1033576, PRJNA1096619, PRJNA224705, PRJNA302221, PRJNA313379, PRJNA358059, PRJNA368916, PRJNA378644, PRJNA382136, PRJNA419306, PRJNA429659, PRJNA475542, PRJNA478998, PRJNA481720 ... (30 more) ]"


In [3]:
// ensure all repos are locally cloned

let base_path = "G:/source/dataplant-gitlab/INSDC_Curation"

let repo_paths =
    df
    |> Frame.mapRows (fun rk os ->
        let datahub_url = os.GetAs<string>("URL")
        let project_accession = os.GetAs<string>("Project_Accession")
        Path.Combine(base_path, project_accession)
    )
    |> Series.values
    |> Seq.toList

let df_with_repos =
    df
    |> fun f ->
        let repo_path_col = 
            f
            |> Frame.mapRows (fun rk os ->
                let datahub_url = os.GetAs<string>("URL")
                let project_accession = os.GetAs<string>("Project_Accession")
                let repo_path = Path.Combine(base_path, project_accession)
                if not (System.IO.Directory.Exists(repo_path)) then
                    printfn $"Cloning {project_accession}..."
                    let git_clone_cmd = $"git clone {datahub_url} {repo_path}"
                    let proc = System.Diagnostics.Process.Start("cmd.exe", $"/C {git_clone_cmd}")
                    proc.WaitForExit()
                    repo_path
                else
                    repo_path
            )
        f
        |> Frame.addCol "Repo_Path" repo_path_col


In [4]:
df_with_repos

0,->,PRJEB25079,6986,ERP106963,https://dee2.io/huge/athaliana/ERP106963_NA.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJEB25079,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079
1,->,PRJNA358059,2516,SRP095347,https://dee2.io/huge/athaliana/SRP095347_GSE92568.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA358059,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059
2,->,PRJNA382136,1407,SRP103736,https://dee2.io/huge/athaliana/SRP103736_GSE97500.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA382136,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136
3,->,PRJNA475542,2359,SRP150217,https://dee2.io/huge/athaliana/SRP150217_GSE115583.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA475542,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542
4,->,PRJNA483458,1732,SRP155742,https://dee2.io/huge/athaliana/SRP155742_GSE117857.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA483458,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458
:,,...,...,...,...,...,...,...,...,...,...
45,->,PRJNA871888,1114,SRP393237,https://dee2.io/huge/athaliana/SRP393237_GSE211718.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA871888,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA871888
46,->,PRJNA906171,3067,SRP410309,https://dee2.io/huge/athaliana/SRP410309_GSE218944.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA906171,True,True,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA906171
47,->,PRJNA938512,976,SRP424504,https://dee2.io/huge/athaliana/SRP424504_GSE226105.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA938512,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA938512
48,->,PRJNA945404,6909,SRP427656,https://dee2.io/huge/athaliana/SRP427656_GSE227500.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA945404,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA945404
49,->,PRJNA989636,2207,SRP446862,https://dee2.io/huge/athaliana/SRP446862_GSE236290.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA989636,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA989636


In [5]:
open ARCtrl
open FsHttp
open System.IO.Compression

let downloadAndExtract (url: string) (targetDir: string) =
    // 1. temp folder for the download
    let tempDir = Path.Combine(Path.GetTempPath(), Path.GetRandomFileName())
    Directory.CreateDirectory(tempDir) |> ignore
    let tempZip = Path.Combine(tempDir, "download.zip")
    try
        // 2. download into temp
        http { GET url }
        |> Request.send
        |> Response.saveFile tempZip

        // 3. unzip into target
        Directory.CreateDirectory(targetDir) |> ignore
        ZipFile.ExtractToDirectory(tempZip, targetDir, overwriteFiles = true)
    finally
        // 4. delete temp (runs even if download/extract throws)
        if Directory.Exists tempDir then Directory.Delete(tempDir, true)

In [6]:
let hasNoSubfolders (path: string) =
    Directory.GetDirectories(path).Length = 0

let init_dee2_assay arc_path dee2_url =

    let arc = ARC.load(arc_path)

    let gitattributes_path = Path.Combine(arc_path, ".gitattributes")

    if not (File.Exists(gitattributes_path)) then
        printfn $"Creating .gitattributes file at {gitattributes_path}..."
        File.WriteAllText(gitattributes_path, "**/dataset/** filter=lfs diff=lfs merge=lfs -text\n")
    else
        printfn $".gitattributes file already exists at {gitattributes_path}, skipping creation."

    let dee2_assay =
        match arc.TryGetAssay("dee2") with
        | Some assay -> 
            assay
        | None ->
            let assay = ArcAssay.init("dee2")
            arc.AddAssay(assay)
            assay

    arc.Update(arc_path)
    let dee2_dataset_path = Path.Combine(arc_path, "assays", "dee2", "dataset")
    let dee2_protocol_path = Path.Combine(arc_path, "assays", "dee2", "protocols")

    // only do this when the folders are empty (except .gitkeep)
    if hasNoSubfolders dee2_dataset_path then
        printfn $"Downloading and extracting DEE2 dataset from {dee2_url} to {dee2_dataset_path}..."
        downloadAndExtract dee2_url dee2_dataset_path
    else
        printfn $"Dataset folder {dee2_dataset_path} is not empty, skipping download."
 
    let protocol_path = Path.Combine(dee2_protocol_path, "dee2_pipeline.md")
    if not (File.Exists(protocol_path)) then
        printfn $"Copying DEE2 protocol to {protocol_path}..."
        File.Copy("./dee2_pipeline.md", protocol_path, true)
    else
        printfn $"Protocol file {protocol_path} already exists, skipping copy."


In [7]:
df_with_repos
|> Frame.mapRows (fun rk os ->
    let repo_path = os.GetAs<string>("Repo_Path")
    let dee2_url = os.GetAs<string>("dee2_url")
    if not (String.IsNullOrWhiteSpace(dee2_url)) then
        init_dee2_assay repo_path dee2_url
)

.gitattributes file already exists at G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079\.gitattributes, skipping creation.
Dataset folder G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079\assays\dee2\dataset is not empty, skipping download.
Protocol file G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079\assays\dee2\protocols\dee2_pipeline.md already exists, skipping copy.
.gitattributes file already exists at G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059\.gitattributes, skipping creation.
Dataset folder G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059\assays\dee2\dataset is not empty, skipping download.
Protocol file G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059\assays\dee2\protocols\dee2_pipeline.md already exists, skipping copy.
.gitattributes file already exists at G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136\.gitattributes, skipping creation.
Dataset folder G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136\assays\dee2\dataset is not empty,

0,->,
1,->,
2,->,
3,->,
4,->,
...,->,...
45,->,
46,->,
47,->,
48,->,
49,->,


In [10]:
open System.IO
open System.Diagnostics

/// Runs a git command inside `repoPath` and returns (exitCode, stdout, stderr).
let runGit (repoPath: string) (args: string) =
    let psi = ProcessStartInfo(FileName = "git", Arguments = args, WorkingDirectory = repoPath)
    psi.RedirectStandardOutput <- true
    psi.RedirectStandardError <- true
    psi.UseShellExecute <- false
    psi.CreateNoWindow <- true
    use proc = Process.Start(psi)
    let stdout = proc.StandardOutput.ReadToEnd()
    let stderr = proc.StandardError.ReadToEnd()
    proc.WaitForExit()
    proc.ExitCode, stdout, stderr

/// Stages, commits and pushes ONLY the DEE2 dataset & protocols (plus the repo's
/// .gitattributes, so the LFS rules travel with the commit), and only when those
/// folders actually contain changes.
///
/// No per-repo `git lfs install` and no pre-push hook: git-lfs is installed globally,
/// so the global clean filter already turns dataset files into LFS pointers at `git add`
/// time. We upload the LFS objects ourselves with an explicit `git lfs push`.
let commitAndPushDee2 (repoPath: string) =
    // Paths allowed into the commit: the two dee2 folders + the root .gitattributes (if present).
    let paths =
        [ "assays/dee2/dataset"
          "assays/dee2/protocols"
          "assays/dee2/README.md"
          "assays/dee2/isa.assay.xlsx"
          if File.Exists(Path.Combine(repoPath, ".gitattributes")) then ".gitattributes" ]
    let pathspec = "-- " + String.concat " " paths

    // 1. Is there anything to commit *in those paths*?
    let _, status, _ = runGit repoPath $"status --porcelain {pathspec}"
    if System.String.IsNullOrWhiteSpace status then
        printfn $"[{repoPath}] No changes in dee2 dataset/protocols - nothing to commit."
    else
        let _, branchOut, _ = runGit repoPath "rev-parse --abbrev-ref HEAD"
        let branch = branchOut.Trim()

        let steps =
            [ "stage",    $"add {pathspec}"
              "commit",   $"commit -m \"Add DEE2 dataset and protocols\" {pathspec}"
              // upload the LFS blobs first (no hook to do it for us), then the commits,
              // so the remote never ends up with pointers whose objects are missing.
              "lfs push", $"lfs push origin {branch}"
              "push",     $"push origin {branch}" ]

        let rec run remaining =
            match remaining with
            | [] ->
                printfn $"[{repoPath}] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/{branch})."
            | (name, args) :: rest ->
                let code, _, err = runGit repoPath args
                if code <> 0 then
                    printfn $"[{repoPath}] git {name} failed (exit {code}): {err.Trim()}"
                else
                    run rest

        run steps


In [11]:
// commit & push the dee2 dataset/protocols for every repo that has them
df_with_repos
|> Frame.mapRows (fun rk os ->
    let repo_path = os.GetAs<string>("Repo_Path")
    printfn $"Processing repo: {repo_path}"
    commitAndPushDee2 repo_path
)

Processing repo: G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079
[G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/main).
Processing repo: G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/main).
Processing repo: G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/main).
Processing repo: G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/main).
Processing repo: G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458] Pushed dee2 dataset/protocols (LFS objects uploaded to origin/main).
Processing repo: G:/so

0,->,
1,->,
2,->,
3,->,
4,->,
...,->,...
45,->,
46,->,
47,->,
48,->,
49,->,
